# 第 10 章: ロジスティック回帰とアンサンブル学習の探索と可視化

ロジスティック回帰の損失の推移と重み、モデル別の特徴量の重要度、森の大きさと正解率を確かめる。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/dotnet/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Microsoft.ML.FastTree, 5.0.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter02.IrisPreprocessing
open MachineLearning.Chapter03
open MachineLearning.Chapter10
open MachineLearning.Chapter10.Classifier
open MachineLearning.Chapter10.FeatureImportance
open MachineLearning.Chapter10.MlNetClassifier

let split = prepareIris (Path.Combine(dataDir (), "iris.csv")) 0.3 0

## モデルごとの正解率

In [ ]:
Main.models
|> List.map (fun (name, classifier) ->
    let score = evaluate classifier split
    {| モデル = name; 訓練データ = score.Train; テストデータ = score.Test |})
|> List.toArray

## ロジスティック回帰の損失の推移

In [ ]:
let logistic = LogisticRegression.fit LogisticRegression.defaults split.XTrain split.TTrain

Chart.Line(x = [ 1 .. logistic.Losses.Length ], y = logistic.Losses)
|> Chart.withTitle "勾配降下法の繰り返し回数と損失"
|> Chart.withXAxisStyle "繰り返し回数"
|> Chart.withYAxisStyle "交差エントロピー"

In [ ]:
[ 1; 10; 100; 1000; 5000 ]
|> List.map (fun epoch -> {| 繰り返し回数 = epoch; 交差エントロピー = logistic.Losses[epoch - 1] |})
|> List.toArray

In [ ]:
logistic.Features
|> List.mapi (fun f feature ->
    {| 特徴量 = feature
       setosa = logistic.Weights[f][0]
       versicolor = logistic.Weights[f][1]
       virginica = logistic.Weights[f][2] |})
|> List.toArray

## モデル別の特徴量の重要度

In [ ]:
let tree = DecisionTree.fit (Some 3) split.XTrain split.TTrain
let forest = RandomForest.fit { RandomForest.defaults with NEstimators = 100 } split.XTrain split.TTrain

let importances =
    [
        "決定木（深さ 3）", treeImportances tree split.XTrain split.TTrain
        "ランダムフォレスト（100 本）", forestImportances forest split.XTrain split.TTrain
    ]

importances
|> List.map (fun (name, values) ->
    Chart.Column(keysValues = (values |> Map.toList), Name = name))
|> Chart.combine
|> Chart.withTitle "モデル別の特徴量の重要度"
|> Chart.withXAxisStyle "特徴量"
|> Chart.withYAxisStyle "重要度"

In [ ]:
let treeValues, forestValues = snd importances[0], snd importances[1]

treeValues
|> Map.toList
|> List.map (fun (feature, value) -> {| 特徴量 = feature; 決定木 = value; ランダムフォレスト = forestValues[feature] |})
|> List.toArray

## 森の大きさと正解率

In [ ]:
let sizes = [ 1; 5; 10; 25; 50; 100 ]

let sizeScores =
    sizes
    |> List.map (fun n ->
        let mine =
            evaluate (ofModel (RandomForest.fit { RandomForest.defaults with NEstimators = n }) RandomForest.predict) split

        let library = evaluate (fastForest n Main.MlNetMinimumExampleCountPerLeaf) split
        n, mine, library)

sizeScores
|> List.map (fun (n, mine, library) ->
    {| 木の数 = n; 自作の訓練データ = mine.Train; 自作のテストデータ = mine.Test; MLNETのテストデータ = library.Test |})
|> List.toArray

In [ ]:
[
    Chart.Line(x = sizes, y = (sizeScores |> List.map (fun (_, mine, _) -> mine.Train)), Name = "自作（訓練データ）", ShowMarkers = true)
    Chart.Line(x = sizes, y = (sizeScores |> List.map (fun (_, mine, _) -> mine.Test)), Name = "自作（テストデータ）", ShowMarkers = true)
    Chart.Line(x = sizes, y = (sizeScores |> List.map (fun (_, _, library) -> library.Test)), Name = "ML.NET（テストデータ）", ShowMarkers = true)
]
|> Chart.combine
|> Chart.withTitle "ランダムフォレストの木の数と正解率"
|> Chart.withXAxisStyle "木の数"
|> Chart.withYAxisStyle "正解率"